In [1]:
import pandas as pd

df = pd.read_csv("../data/hotel_bookings.csv")

# reservation_status/date directly encode the outcome — leakage, must drop
# agent/company dropped for potential leakage; confirmed via testing
# that removing them changes scores by <1%, so no meaningful signal lost
X = df.drop(columns=["is_canceled","reservation_status","reservation_status_date","agent","company"])
y=df["is_canceled"]

In [2]:
#dividing numeric and categorical columns

int_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["str"]).columns.tolist()

In [3]:
#Dropping high cardinality columns that balloon up training data and time with minor returns

high_cardinality_cols = ["country"]
cat_cols = [col for col in cat_cols if col not in high_cardinality_cols]



print("Categorical Columns used:",cat_cols)
print("Numeric Columns used:",int_cols)
print("Number of Categorical Columns:",len(cat_cols))

Categorical Columns used: ['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
Numeric Columns used: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
Number of Categorical Columns: 9


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

#Imputing NaN values in numeric cols
numeric_transform = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

#Imputing empty values then encoding
categorical_transform = Pipeline(steps=[("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))])

#combining both transformation pipelines
preprocessor = ColumnTransformer(transformers=[("num",numeric_transform,int_cols),("category",categorical_transform,cat_cols)])


In [5]:
from sklearn.ensemble import RandomForestClassifier

#RFClassifier model using the preprocessor in pipeline
model_1 = Pipeline(steps=[("preprocessor",preprocessor),("classifier",RandomForestClassifier(random_state=0))])

In [6]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold

#shuffling training data and validation data assigned to each cv
cv_strategy = StratifiedKFold(n_splits=5,shuffle=True,random_state=0)

metrics = ["recall","precision","f1"]


results = cross_validate(model_1,X,y,cv=cv_strategy,scoring=metrics,n_jobs=-1,error_score="raise")

for metric in metrics:
    fold_scores = results[f"test_{metric}"]
    print(f"{metric.capitalize()} per fold:", fold_scores)
    print(f"Mean {metric.capitalize()} per fold:",fold_scores.mean())
    print(f"Standard Deviation of {metric.capitalize()} per fold:",fold_scores.std())


Recall per fold: [0.76797829 0.76517807 0.76551724 0.7697004  0.76992651]
Mean Recall per fold: 0.767660101261512
Standard Deviation of Recall per fold: 0.0020076358269266155
Precision per fold: [0.8702114  0.86791485 0.86740968 0.86472755 0.8697318 ]
Mean Precision per fold: 0.8679990576772749
Standard Deviation of Precision per fold: 0.0019470020494159533
F1 per fold: [0.81590486 0.81331491 0.81328449 0.81445149 0.8167916 ]
Mean F1 per fold: 0.8147494694695123
Standard Deviation of F1 per fold: 0.0013998695000428801


In [7]:
#Baseline: RandomForestClassifier (default params) with median/most-frequent imputation and one-hot encoding. Dropped agent, company, and country 
#— originally excluded to reduce one-hot encoding width and training time (all three are high-cardinality, ballooning the feature space significantly), 
#then kept out after also confirming agent/company weren't acting as a hidden ID-proxy confound (removing them changed scores by <1%, so their earlier 
#presence wasn't inflating performance meaningfully). Also dropped reservation_status/reservation_status_date (direct leakage). 5-fold stratified shuffled
#CV gives recall 0.767, precision 0.867, F1 0.814. This is the number to beat — any future model change needs to justify itself against this.